In [3]:
import pandas as pd

from phd_project.config import config


cfg = config.load_config()

Combines the lists of records to download together into a single csv file for each database

In [4]:
# paths to record lists:
esm_records_AvgSA03_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_03_summary_esm_records_selected.csv"
ngasub_records_AvgSA03_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_03_summary_ngasub_records_selected.csv"
esm_records_AvgSA06_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_06_summary_esm_records_selected.csv"
ngasub_records_AvgSA06_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_06_summary_ngasub_records_selected.csv"

esm_combined_fp = cfg["proc_data"]["gm_selection"] / f"all_esm_records_to_download.csv"
ngasub_combined_fp = cfg["proc_data"]["gm_selection"] / f"all_ngasub_records_to_download.csv"

# load the as series and dataframes
esm_06 = pd.read_csv(esm_records_AvgSA06_fp, header=[0, 1])
esm_03 = pd.read_csv(esm_records_AvgSA03_fp, header=[0, 1])
esm_combined = pd.concat([esm_06, esm_03], ignore_index=True).drop_duplicates()
esm_combined.to_csv(esm_combined_fp, index=False)

nga_06 = pd.read_csv(ngasub_records_AvgSA06_fp, header=0)
nga_03 = pd.read_csv(ngasub_records_AvgSA03_fp, header=0)
nga_combined = pd.concat([nga_06, nga_03])
nga_combined = nga_combined.groupby("NGAsubRSN")["count"].sum().reset_index().sort_values("count", ascending=False)
nga_combined.to_csv(ngasub_combined_fp, index=False)

In [5]:
# paths to per-campaign conversion lists:
convert_AvgSA03_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_03_records_to_convert.csv"
convert_AvgSA06_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_06_records_to_convert.csv"

esm_convert_fp = cfg["proc_data"]["gm_selection"] / f"esm_records_to_convert.csv"
ngasub_convert_fp = cfg["proc_data"]["gm_selection"] / f"ngasub_records_to_convert.csv"

conv_03 = pd.read_csv(convert_AvgSA03_fp, dtype=str)
conv_06 = pd.read_csv(convert_AvgSA06_fp, dtype=str)
conv_combined = pd.concat([conv_03, conv_06], ignore_index=True).drop_duplicates()

for db_name, fp in [("ESM", esm_convert_fp), ("NGASub", ngasub_convert_fp)]:
    sub = (conv_combined[conv_combined["database"] == db_name]
           [["record_identifier", "component"]]
           .drop_duplicates()
           .reset_index(drop=True))
    sub.to_csv(fp, index=False)
    print(f"{fp.name}: {len(sub)} records")


esm_records_to_convert.csv: 1874 records
ngasub_records_to_convert.csv: 2137 records
